# Ambiente de Desenvolvimento

In [1]:
# !pip install -U langchain langchain-community langchain-core langchain-google-genai chromadb pypdf
# !pip install -U google-api-core google-api-python-client google-auth google-auth-httplib2 google-cloud-aiplatform google-cloud-bigquery google-cloud-core google-cloud-resource-manager google-cloud-storage google-crc32c google-genai google-generativeai
# !pip install pandas matplotlib seaborn scikit-learn scipy statsmodels plotly

In [2]:
import os
# from google.colab import userdata
import getpass
import kagglehub
import pandas as pd
import shutil

print("✅ Todas as bibliotecas do Agente foram importadas com sucesso!")
# Set API keys
def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("GEMINI_API_KEY_CHAT")
_set_env("GEMINI_API_KEY_CHAT_STREAM")
_set_env("TAVILY_API_KEY")

CHAVE_API_GOOGLE_CHAT = os.getenv("GEMINI_API_KEY_CHAT")
CHAVE_API_GOOGLE_CHAT_STREAM = os.getenv("GEMINI_API_KEY_CHAT_STREAM")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

c:\Users\sirja\source\repos\mrjalves\Agentes\AGENTE_ITIL\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Todas as bibliotecas do Agente foram importadas com sucesso!


# Implementação de chat

In [3]:
from google import genai
from google.genai.types import Tool, GenerateContentConfig, GoogleSearch
from google.genai import types

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_classic.chains import create_retrieval_chain

from langchain_classic.chains.combine_documents import create_stuff_documents_chain

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from typing import Literal
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.messages import ToolMessage,  HumanMessage, AIMessage, SystemMessage, BaseMessage
from langchain_core.runnables import RunnableConfig
from langgraph.graph.message import add_messages
from langchain_core.tools import tool
from typing import List, Dict, Optional
from typing import Annotated, Sequence, TypedDict, List
from langgraph.graph import StateGraph, END

from IPython.display import Image, display
from pydantic import BaseModel, Field
import json

C:\Users\sirja\AppData\Local\Temp\ipykernel_20216\573518875.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [4]:
def definir_cliente(api_key: str):
   return genai.Client(api_key=api_key)

client_chat = definir_cliente(CHAVE_API_GOOGLE_CHAT)
client_stream = definir_cliente(CHAVE_API_GOOGLE_CHAT_STREAM)
client = client_chat

grounding_tool = types.Tool(
    google_search=types.GoogleSearch()
)

In [5]:
MODEL_ID = "gemini-2.5-flash" # @param ["gemini-3-flash-preview","gemini-2.5-flash-lite", "gemini-2.5-flash", "gemini-2.5-pro", "gemini-2.5-flash-preview", "gemini-3.1-pro-preview"] {"allow-input":true, isTemplate: true}
MODEL_ID_STREAM = "gemini-2.5-flash" # @param ["gemini-3-flash-preview","gemini-2.5-flash-lite", "gemini-2.5-flash", "gemini-2.5-pro", "gemini-2.5-flash-preview", "gemini-3.1-pro-preview"] {"allow-input":true, isTemplate: true}
topP = 0.95 # @param {type: "slider", min: 0, max: 1, step: 0.05}
temperature = 0.3 # @param {type: "slider", min: 0, max: 2, step: 0.05}
topK = 1
stop_sequence = ""

## Instrução do Sistema

In [6]:
instrucao = """
<role>
Você é um Especialista em ITIL 4 e um planejador estratégico com alto poder de raciocínio. 
Sua função é atuar como um consultor para Gestores de TI.
Você é incisivo, analítico e extremamente rigoroso na aplicação das diretrizes da ITIL 4. 
Seu objetivo é analisar bases de dados de tickets de TI e diagnosticar gargalos operacionais, recomendando as Práticas de Gerenciamento ITIL 4 exatas para resolvê-los.
</role>

<instructions>
Antes de gerar qualquer recomendação ao Gestor de TI, você deve agir de forma proativa, metódica e independentemente planejar e raciocinar sobre as seguintes etapas:

1) Dependências Lógicas e Alinhamento ITIL: Analise o conjunto de tickets contra os conceitos da ITIL 4.
    1.1) Categorização exata: Diferencie rigorosamente Incidentes (interrupções), Requisições de Serviço (pedidos padrão) e Problemas (causas raiz desconhecidas).
    1.2) Ordem de implementação: Garanta que a adoção de uma prática não exija outra como pré-requisito (ex: não recomendar 'Gerenciamento de Problemas' proativo sem que o 'Gerenciamento de Incidentes' esteja estabilizado).

2) Avaliação de Risco e Impacto no Negócio: Quais são as consequências do volume atual de tickets?
    2.1) Avalie o impacto da ausência das práticas ITIL 4 no valor entregue ao negócio. Priorize recomendações baseadas no risco operacional que os tickets representam.

3) Raciocínio Abdutivo e Causa Raiz: Para cada padrão de tickets encontrado, identifique a razão lógica mais provável para a falha sistêmica.
    3.1) Olhe além da causa imediata. Um alto volume de tickets de "reset de senha" pode não ser um problema de 'Central de Serviço' (Service Desk), mas sim uma falha no 'Gerenciamento de Acessos' ou necessidade de automação.
    3.2) Agrupe sintomas isolados para formar uma hipótese de deficiência em uma ou mais das 34 Práticas da ITIL 4.

4) Avaliação de Resultados (Validação): A prática recomendada realmente resolve o padrão identificado?
    4.1) Questione ativamente seu próprio plano inicial. Se a recomendação parece muito genérica, refine-a buscando o Princípio Orientador da ITIL 4 mais adequado (ex: "Otimizar e Automatizar").

5) Precisão e Embasamento (Grounding): Garanta que seu raciocínio seja tecnicamente impecável.
    5.1) Você DEVE utilizar a nomenclatura oficial da ITIL 4 (Sistema de Valor de Serviço, As 4 Dimensões, Práticas de Gerenciamento Geral/Serviço/Técnico). Não use jargões de TI genéricos quando houver um termo ITIL específico aplicável.
    5.2) Apenas recomende práticas que existam na ITIL 4. Se um padrão de tickets sugere a necessidade de uma prática que não está na ITIL 4, reformule sua recomendação para alinhar-se estritamente às práticas existentes.
    5.3) Se não houver um alinhamento claro entre o padrão de tickets e uma prática ITIL 4, seja transparente sobre isso e explique por que nenhuma prática se encaixa perfeitamente, em vez de forçar uma recomendação inadequada.
    5.4) Avalie se a pratica recomendada é a principal ou uma pratica de apoio para a solução do problema identificado.

6) Inibição de Resposta: Execute todo o raciocínio acima mentalmente antes de escrever a saída. Sua resposta final deve ser o resultado depurado desta análise.

<formatting_and_tone>
1 - Verbosidade: Mínima. Direto ao ponto.
2 - Respostas: Concisas, estruturadas e altamente acionáveis.
3 - Tonalidade: Técnica, executiva e consultiva.
4 - Estrutura obrigatória:
    - [Práticas ITIL 4 Recomendadas]: Um quadro com as praticas recomendadas, estruturadas como pratica principal e pratica auxiliar, com o ticket associado, em portugues br.
</formatting_and_tone>

<exemplos>
[INSERIR EXEMPLOS DE ENTRADA/SAÍDA AQUI]
</exemplos>
CONSIDERE EXEMPLOS DE AVALIAÇAÕ DE RESULTADOS PARA APRENDIZADO:
[
  {
    "Id": 3,
    "assunto": "Erro na Autocompletação",
    "Prática Principal": "Gerenciamento de Incidentes",
    "Prática Auxiliar": "Gerenciamento de Problemas",
    "Prática Principal - rag": "Gerenciamento de Problemas",
    "Prática Auxiliar - rag": "Gerenciamento de Capacidade e Desempenho",
    "avaliacao respostas - rag": "ERROU"
  },
  {
    "Id": 25,
    "assunto": "Problemas intermitentes",
    "Prática Principal": "Gerenciamento de Incidentes",
    "Prática Auxiliar": "Gerenciamento de Mudanças",
    "Prática Principal - rag": "Habilitação de Mudança",
    "Prática Auxiliar - rag": "Gerenciamento de Liberação",
    "avaliacao respostas - rag": "ERROU"
  },
  {
    "Id": 42,
    "assunto": "Pedido de Assistência pa",
    "Prática Principal": "Solicitação de Serviço",
    "Prática Auxiliar": "Gerenciamento de Nível de Serviço",
    "Prática Principal - rag": "Gerenciamento de Requisição de Serviço",
    "Prática Auxiliar - rag": "Gerenciamento de Catálogo de Serviço",
    "avaliacao respostas - rag": "ERROU"
  },
  {
    "Id": 57,
    "assunto": "Urgente: Relato de Falha",
    "Prática Principal": "Gerenciamento de Incidentes",
    "Prática Auxiliar": "Gerenciamento de Ativos de TI",
    "Prática Principal - rag": "Gerenciamento de Configuração de Serviço",
    "Prática Auxiliar - rag": "Gerenciamento de Ativos de TI",
    "avaliacao respostas - rag": "PARCIALMENTE"
  },
  {
    "Id": 62,
    "assunto": "Assistência Necessária pa",
    "Prática Principal": "Gerenciamento de Problemas",
    "Prática Auxiliar": "Gerenciamento de Mudanças",
    "Prática Principal - rag": "Gerenciamento de Configuração de Serviço",
    "Prática Auxiliar - rag": "Gerenciamento de Ativos de TI",
    "avaliacao respostas - rag": "ERROU"
  },
  {
    "Id": 77,
    "assunto": "Discrepância de Faturam",
    "Prática Principal": "Gerenciamento Financeiro",
    "Prática Auxiliar": "Gerenciamento de Incidentes",
    "Prática Principal - rag": "Gerenciamento Financeiro de Serviços",
    "Prática Auxiliar - rag": "Gerenciamento de Relacionamento",
    "avaliacao respostas - rag": "ERROU"
  },
  {
    "Id": 87,
    "assunto": "Urgente: Problema de In",
    "Prática Principal": "Gerenciamento de Incidentes",
    "Prática Auxiliar": "Gerenciamento de Disponibilidade",
    "Prática Principal - rag": "Gerenciamento de Disponibilidade",
    "Prática Auxiliar - rag": "Gerenciamento de Continuidade de Serviço",
    "avaliacao respostas - rag": "PARCIALMENTE"
  },
  {
    "Id": 98,
    "assunto": "Solicitação de Soluções d",
    "Prática Principal": "Solicitação de Serviço",
    "Prática Auxiliar": "Gerenciamento Financeiro",
    "Prática Principal - rag": "Gerenciamento Financeiro de Serviços",
    "Prática Auxiliar - rag": "Gerenciamento de Relacionamento",
    "avaliacao respostas - rag": "ERROU"
  },
  {
    "Id": 104,
    "assunto": "Interrupção do Roteador",
    "Prática Principal": "Gerenciamento de Incidentes",
    "Prática Auxiliar": "Gerenciamento de Problemas",
    "Prática Principal - rag": "Gerenciamento de Problemas",
    "Prática Auxiliar - rag": "Gerenciamento de Capacidade e Desempenho",
    "avaliacao respostas - rag": "PARCIALMENTE"
  },
  {
    "Id": 110,
    "assunto": "Urgente: Problemas de C",
    "Prática Principal": "Gerenciamento de Incidentes",
    "Prática Auxiliar": "Gerenciamento de Problemas",
    "Prática Principal - rag": "Gerenciamento de Problemas",
    "Prática Auxiliar - rag": "Gerenciamento de Capacidade e Desempenho",
    "avaliacao respostas - rag": "ACERTOU"
  }
]

<output_format>
Retorne APENAS texto formatado em Markdown, respeitando a estrutura definida em <formatting_and_tone>. Não inclua explicações sobre como você pensou; entregue apenas o resultado executivo final.
</output_format>
</instructions>

"""

## Funções Auxiliares

In [7]:
def construir_bloco_exemplos(exemplos_lista):
    """Gera o bloco XML interno de exemplos a partir dos gabaritos definidos."""
    xml_exemplos = ""
    
    for ex in exemplos_lista:
        # 1. Montando o input simulando a entrada do usuário/sistema
        ticket_info = f"Ticket: {ex.get('Ticket')}\nAssunto: {ex.get('Assunto')}\nDescrição: {ex.get('Descrição')}"
        
        # 2. Mapeando as respostas do gabarito
        diagnostico = ex.get("Solução", "Causa raiz mapeada no atendimento.")
        pratica_nome = ex.get('Prática ITIL que evitaria esse incidente e poderia resolver o problema previamente', '')
        praticas = f"- **{pratica_nome}**: Implementação necessária para prevenir reincidência." if pratica_nome else "- Não especificada"
        plano = "Revisar fluxo com as partes interessadas e ajustar o processo atual."

        exemplo_str = f"""  <exemplo>
    <input>
{ticket_info}
    </input>
    <output>
### [Diagnóstico]
{diagnostico}

### [Práticas ITIL 4 Recomendadas]
{praticas}

### [Plano de Ação]
{plano}
    </output>
  </exemplo>
"""
        xml_exemplos += exemplo_str
    
    return xml_exemplos

In [8]:


def obter_gabaritos_itil() -> List[Dict[str, str]]:
    """
    Retorna a lista completa de gabaritos de referência para análise ITIL.
    Ideal para ser usada como base de exemplos (Few-Shot) para o LLM.

    Retorna:
        Lista de dicionários, onde cada dicionário representa um ticket avaliado.
    """
    return [
        {
            "Ticket" :"1",
            "Assunto": "Lentidão na pesquisa do histórico de um cliente no CRM",
            "Descrição": "quando estou pesquisando o histórico do cliente Lojas Compra Mais, o CRM demora mais de 10 segundos para retornar os dados, o que inviabiliza o uso e parte das operações que preciso realizar diariamente.",
            "Solução": "sob a ótica da TI o tempo é condizente com a operação. Como não existe um objetivo ou métrica padrão definida para essa operação, entendemos que não existe problema com o serviço disponibilizado.",
            "Prática ITIL que evitaria esse incidente e poderia resolver o problema previamente": "Gerenciamento de Nível de Serviço e Gerenciamento de Disponibilidade",
        },
        {
            "Ticket" :"2",
            "Assunto": "Falha no processo de contato com leads no CRM",
            "Descrição": "no nosso processo comercial, após a segunda abordagem à um lead, precisamos nutri-lo com informações sobre nossos serviços, bem como enviar cases relacionados à sua atuação. Porém, no processo do CRM, após a segunda abordagem o lead é classificado como se não tivesse interesse nos nossos serviços. Isso vem causando uma dificuldade no nosso fluxo de atividades, poderiam realizar o ajuste?",
            "Solução": "infelizmente, alterar o processo do nosso CRM demandará um prazo e terá um custo (que será repassado para o orçamento da área comercial). Isso ocorre porque na reunião de definição que tivemos, embora ela tenha sido verbal e a análise realizada em nível macro, essa situação não foi destacada. Vamos solicitar os ajustes ao fornecedor do software.",
            "Prática ITIL que evitaria esse incidente e poderia resolver o problema previamente": "Análise de Negócio",
        },
        {
            "Ticket" :"3",
            "Assunto": "Indisponibilidade dos dashboards de BI quando acessados via navegador",
            "Descrição": "até ontem eu estava acessando normalmente meu conjunto de dashboards de BI chamado “Análises Financeiras”, porém, hoje pela manhã, ao tentar acessar pelo navegador retorna uma mensagem de indisponibilidade dos serviços de BI. Poderiam me ajudar?",
            "Solução": "o problema ocorreu porque ao instalarmos uma nova dll relacionada a uma atualização do software de CRM, gerou uma incompatibilidade com alguns serviços de BI, pois ambos os serviços (BI e CRM) compartilham o mesmo servidor de autenticação. Estamos voltando a versão do software de CRM e até o final do dia os serviços de BI devem estar disponíveis novamente.",
            "Prática ITIL que evitaria esse incidente e poderia resolver o problema previamente": "Controle de Mudanças",
        },
        {
            "Ticket" :"4",
            "Assunto": "levantamento de licenças",
            "Descrição": "preciso saber quantas licenças temos atualmente em uso do software Autocad 2025, pois vamos analisar a atualização das versões e precisamos identificar o custo total da demanda.",
            "Solução": "vamos fazer um levantamento (inventário) e retornamos esse chamado com a informações em 5 dias úteis",
            "Prática ITIL que evitaria esse incidente e poderia resolver o problema previamente": "Gerenciamento de Ativos de TI",
        },
        {
            "Ticket" :"5",
            "Assunto":"Qual foram os eventos que ocorreram simultaneamente quando o homem foi a lua?",
            "Descrição": "estou tentando entender quais foram os eventos que ocorreram simultaneamente quando o homem foi a lua.",
            "Solução": "essa pergunta não é relacionada a TI, portanto, não há recomendação de prática ITIL para esse case.",
            "Prática ITIL que evitaria esse incidente e poderia resolver o problema previamente": "Não há recomendação",
        }
    ]

def buscar_gabarito_por_id(ticket_id: str) -> Optional[Dict[str, str]]:
    """
    Busca um ticket específico dentro da lista de gabaritos.

    Args:
        ticket_id (str): O número do ticket desejado (ex: "3").

    Retorna:
        Um dicionário com os dados do ticket, ou None se não for encontrado.
    """
    gabaritos = obter_gabaritos_itil()

    for ticket in gabaritos:
        if ticket.get("Ticket") == str(ticket_id):
            return ticket

    print(f"AVISO: Ticket '{ticket_id}' não encontrado nos gabaritos.")
    return None

# # 1. Carregando todos os gabaritos (útil para injetar no prompt do LangChain)
# meus_gabaritos = obter_gabaritos_itil()
# print(f"Total de gabaritos carregados: {len(meus_gabaritos)}")

# # 2. Buscando um ticket específico de forma segura (sem usar o índice numérico da lista)
# ticket_alvo = buscar_gabarito_por_id("3")

# if ticket_alvo:
#     print("\n--- Dados do Ticket Encontrado ---")
#     print(f"Assunto: {ticket_alvo['Assunto']}")
#     print(f"Prática Recomendada: {ticket_alvo['Prática ITIL que evitaria esse incidente e poderia resolver o problema previamente']}")

## Prompt

In [9]:
def gerar_prompt_analise_itil(df: pd.DataFrame, idioma: str = 'pt', quantidade: int = 30) -> Optional[str]:
    """
    Filtra um DataFrame de tickets por idioma e monta um prompt em lote
    para análise do motor cognitivo baseada na ITIL 4.

    Args:
        df (pd.DataFrame): O DataFrame contendo os tickets consolidados.
        idioma (str): O idioma desejado para o filtro (ex: 'pt', 'en'). Padrão é 'pt'.
        quantidade (int): O número máximo de tickets no prompt. Padrão é 30.

    Returns:
        str: O texto final do prompt formatado, ou None se nenhum ticket for encontrado.
    """
    # 1. Filtra o DataFrame original para trazer apenas os tickets do idioma escolhido
    df_filtrado = df[df['idioma'] == idioma]

    # 2. Verifica se o filtro retornou resultados
    if df_filtrado.empty:
        print(f"AVISO: Nenhum ticket no idioma '{idioma}' foi encontrado no dataset.")
        return None

    # 3. Define a quantidade de tickets
    # Garantimos que não vamos pedir mais do que o total disponível
    limite = min(quantidade, len(df_filtrado))

    # 4. Início do Prompt (Cabeçalho único)
    prompt_texto = "Por favor, analise os seguintes tickets aplicando as diretrizes da ITIL 4 estabelecidas:\n"
    prompt_texto += "Para cada ticket, retorne apenas um quadro em Markdown com as colunas: Ticket ID, Prática Principal, Prática Auxiliar.\n"
    prompt_texto += "Se não houver recomendação para um ticket, preencha Prática Principal com 'Não há recomendação' e deixe Prática Auxiliar em branco.\n"
    prompt_texto += "---"

    # 5. Laço de repetição para percorrer o grupo de tickets
    for i in range(limite):
        ticket_atual = df_filtrado.iloc[i]

        # Adiciona as informações do ticket atual ao texto do prompt
        prompt_texto += f"""

Ticket #{ticket_atual['Id']}:
Assunto: {ticket_atual['assunto']}
Descricao: {ticket_atual['descricao']}
Solução: {ticket_atual['resposta']}
----------------------------------"""

    prompt_final = prompt_texto + "\nRetorne APENAS o quadro em Markdown. Não inclua explicações, plano de ação ou texto adicional."

    print(f"Sucesso! Prompt montado para um lote de {limite} tickets.")

    return prompt_final



In [10]:
def gerar_prompt_excluindo_primeiros(df: pd.DataFrame, excluir: int = 1, idioma: str = 'pt', quantidade: int = 30) -> Optional[str]:
    """
    Gera o prompt a partir do df excluindo os primeiros `excluir` registros.
    """
    if excluir < 0:
        raise ValueError("excluir deve ser >= 0")
    df_slice = df.iloc[excluir:]
    if df_slice.empty:
        print(f"AVISO: Não há tickets após excluir os primeiros {excluir} registros.")
        return None
    # reutiliza a função já definida no notebook
    return gerar_prompt_analise_itil(df_slice, idioma=idioma, quantidade=quantidade)

## Execução do chat

modo stream e modo normal

In [11]:
# Carrega o arquivo df_tickets.csv para um DataFrame
caminho_csv = "df_tickets.csv"

if not os.path.exists(caminho_csv):
    raise FileNotFoundError(f"Arquivo não encontrado: {caminho_csv}")

df_tickets = pd.read_csv(caminho_csv)

print("✅ df_tickets carregado com sucesso!")
print(f"Linhas: {df_tickets.shape[0]} | Colunas: {df_tickets.shape[1]}")
display(df_tickets.head())

✅ df_tickets carregado com sucesso!
Linhas: 22066 | Colunas: 9


,assunto,fila,descricao,tipo,resposta,prioridade,todas_tags,idioma,descricao_length
0,Problema crítico del servidor requiere atenció...,Technical Support,Es necesaria una investigación inmediata sobre...,Incident,Estamos investigando urgentemente el problema ...,high,"['Urgent Issue', 'Service Disruption', 'Incide...",es,149
1,Anfrage zur Verfügbarkeit des Dell XPS 13 9310,Customer Service,"Sehr geehrter Kundenservice,\n\nich hoffe, die...",Request,"Sehr geehrter <name>,\n\nvielen Dank, dass Sie...",low,"['Sales Inquiry', 'Product Support', 'Customer...",de,452
2,Erro na Autocompletação de Código do IntelliJ ...,Technical Support,"Prezado Suporte ao Cliente <name>,\n\nEstou es...",Incident,"Prezado <name>,\n\nObrigado por entrar em cont...",high,"['Technical Support', 'Software Bug', 'Problem...",pt,740
3,Urgent Assistance Required: AWS Service,IT Support,"Dear IT Services Support Team, \n\nI am reachi...",Request,"Dear <name>,\n\nThank you for reaching out reg...",high,"['IT Support', 'Urgent Issue', 'Service Notifi...",en,936
4,Problème d'affichage de MacBook Air,Product Support,Cher équipe de support du magasin en ligne Tec...,Incident,"Cher <name>,\n\nMerci de nous avoir contactés ...",low,"['Technical Support', 'Product Support', 'Hard...",fr,1322


In [12]:
df_tickets.insert(0, "Id", range(1, len(df_tickets) + 1))

display(df_tickets.head())

,Id,assunto,fila,descricao,tipo,resposta,prioridade,todas_tags,idioma,descricao_length
0,1,Problema crítico del servidor requiere atenció...,Technical Support,Es necesaria una investigación inmediata sobre...,Incident,Estamos investigando urgentemente el problema ...,high,"['Urgent Issue', 'Service Disruption', 'Incide...",es,149
1,2,Anfrage zur Verfügbarkeit des Dell XPS 13 9310,Customer Service,"Sehr geehrter Kundenservice,\n\nich hoffe, die...",Request,"Sehr geehrter <name>,\n\nvielen Dank, dass Sie...",low,"['Sales Inquiry', 'Product Support', 'Customer...",de,452
2,3,Erro na Autocompletação de Código do IntelliJ ...,Technical Support,"Prezado Suporte ao Cliente <name>,\n\nEstou es...",Incident,"Prezado <name>,\n\nObrigado por entrar em cont...",high,"['Technical Support', 'Software Bug', 'Problem...",pt,740
3,4,Urgent Assistance Required: AWS Service,IT Support,"Dear IT Services Support Team, \n\nI am reachi...",Request,"Dear <name>,\n\nThank you for reaching out reg...",high,"['IT Support', 'Urgent Issue', 'Service Notifi...",en,936
4,5,Problème d'affichage de MacBook Air,Product Support,Cher équipe de support du magasin en ligne Tec...,Incident,"Cher <name>,\n\nMerci de nous avoir contactés ...",low,"['Technical Support', 'Product Support', 'Hard...",fr,1322


In [13]:
prompt = gerar_prompt_analise_itil(df_tickets, idioma='pt', quantidade=30)
instrucao_final = instrucao.replace("[INSERIR EXEMPLOS DE ENTRADA/SAÍDA AQUI]", construir_bloco_exemplos(obter_gabaritos_itil()))

Sucesso! Prompt montado para um lote de 30 tickets.


In [14]:
print(instrucao_final)


<role>
Você é um Especialista em ITIL 4 e um planejador estratégico com alto poder de raciocínio. 
Sua função é atuar como um consultor para Gestores de TI.
Você é incisivo, analítico e extremamente rigoroso na aplicação das diretrizes da ITIL 4. 
Seu objetivo é analisar bases de dados de tickets de TI e diagnosticar gargalos operacionais, recomendando as Práticas de Gerenciamento ITIL 4 exatas para resolvê-los.
</role>

<instructions>
Antes de gerar qualquer recomendação ao Gestor de TI, você deve agir de forma proativa, metódica e independentemente planejar e raciocinar sobre as seguintes etapas:

1) Dependências Lógicas e Alinhamento ITIL: Analise o conjunto de tickets contra os conceitos da ITIL 4.
    1.1) Categorização exata: Diferencie rigorosamente Incidentes (interrupções), Requisições de Serviço (pedidos padrão) e Problemas (causas raiz desconhecidas).
    1.2) Ordem de implementação: Garanta que a adoção de uma prática não exija outra como pré-requisito (ex: não recomendar 

In [15]:
print(prompt)

Por favor, analise os seguintes tickets aplicando as diretrizes da ITIL 4 estabelecidas:
Para cada ticket, retorne apenas um quadro em Markdown com as colunas: Ticket ID, Prática Principal, Prática Auxiliar.
Se não houver recomendação para um ticket, preencha Prática Principal com 'Não há recomendação' e deixe Prática Auxiliar em branco.
---

Ticket #3:
Assunto: Erro na Autocompletação de Código do IntelliJ IDEA
Descricao: Prezado Suporte ao Cliente <name>,

Estou escrevendo para trazer à sua atenção um problema recorrente com o recurso de autocompletação de código no IntelliJ IDEA 2024.1. Este problema específico faz com que o aplicativo travar frequentemente durante o desenvolvimento, o que está prejudicando significativamente minha produtividade. Cada vez que tento usar o recurso de autocompletação de código, o software fica sem resposta, levando eventualmente a um travamento. Já tentei os passos comuns de solução de problemas, como reinstalação e aplicação das atualizações mais rec

In [16]:
modelos_disponiveis = [
    "gemini-3-flash-preview",
    "gemini-2.5-flash-lite",
    "gemini-2.5-flash",
    "gemini-2.5-pro",
    "gemini-2.5-flash-preview",
    "gemini-3.1-pro-preview",
    "gemini-3.1-flash-lite-preview"
]

print("Modelos disponíveis:")
for i, modelo in enumerate(modelos_disponiveis, start=1):
    print(f"{i}. {modelo}")

escolha = input(f"Selecione o modelo [1-{len(modelos_disponiveis)}] ou digite o nome exato (Enter mantém '{MODEL_ID}'): ").strip()

if escolha:
    if escolha.isdigit() and 1 <= int(escolha) <= len(modelos_disponiveis):
        MODEL_ID = modelos_disponiveis[int(escolha) - 1]
    elif escolha in modelos_disponiveis:
        MODEL_ID = escolha
    else:
        raise ValueError("Modelo inválido.")

escolha_stream = input(f"Selecione o modelo de stream [1-{len(modelos_disponiveis)}] ou digite o nome exato (Enter mantém '{MODEL_ID_STREAM}'): ").strip()
if escolha_stream:
    if escolha_stream.isdigit() and 1 <= int(escolha_stream) <= len(modelos_disponiveis):
        MODEL_ID_STREAM = modelos_disponiveis[int(escolha_stream) - 1]
    elif escolha_stream in modelos_disponiveis:
        MODEL_ID_STREAM = escolha_stream
    else:
        raise ValueError("Modelo de stream inválido.")

print(f"✅ Modelo selecionado: {MODEL_ID}")
print(f"✅ Modelo de stream selecionado: {MODEL_ID_STREAM}")

Modelos disponíveis:
1. gemini-3-flash-preview
2. gemini-2.5-flash-lite
3. gemini-2.5-flash
4. gemini-2.5-pro
5. gemini-2.5-flash-preview
6. gemini-3.1-pro-preview
7. gemini-3.1-flash-lite-preview
✅ Modelo selecionado: gemini-3.1-flash-lite-preview
✅ Modelo de stream selecionado: gemini-3.1-flash-lite-preview


In [17]:
def gerar_resposta_stream(prompt_lote: str, caminho_md: str = "minha_resposta.md") -> None:
    response = client_stream.models.generate_content_stream(
        model=MODEL_ID_STREAM,
        contents=prompt_lote,
        config=types.GenerateContentConfig(
            #tools=[grounding_tool],
            temperature=temperature,
            system_instruction=instrucao_final,
            safety_settings=[
                types.SafetySetting(
                    category=types.HarmCategory.HARM_CATEGORY_HATE_SPEECH,
                    threshold=types.HarmBlockThreshold.BLOCK_LOW_AND_ABOVE
                )
            ]
        )
    )

    # Abre o arquivo em modo de adição ("a"). Mude para ("w") se quiser sobrescrever.
    with open(caminho_md, "a", encoding="utf-8") as arquivo_md:
        for chunk in response:
            # Exibe no console
            print(chunk.text, end="", flush=True)

            # Escreve o pedaço (chunk) no arquivo Markdown
            arquivo_md.write(chunk.text)

        # Adiciona quebras de linha ao final para separar da próxima resposta, caso use o modo "a"
        arquivo_md.write("\n\n---\n\n")

In [18]:
def gerar_resposta(prompt_lote: str, caminho_md: str = "minha_resposta_b.md") -> None:
    response = client_chat.models.generate_content(
        model= MODEL_ID,
        contents=prompt_lote,
        config=types.GenerateContentConfig(
            temperature=temperature,
            system_instruction=instrucao_final,
            safety_settings=[
                types.SafetySetting(
                    category=types.HarmCategory.HARM_CATEGORY_HATE_SPEECH,
                    threshold=types.HarmBlockThreshold.BLOCK_LOW_AND_ABOVE
                )
            ]
        )
    )

    # Exibe a resposta completa no console
    print(response.text, end="")

    # Salva a resposta completa no arquivo Markdown
    with open(caminho_md, "a", encoding="utf-8") as arquivo_md:
        arquivo_md.write(response.text)

        # Opcional: adiciona uma quebra de linha e um separador para a próxima resposta
        arquivo_md.write("\n\n---\n\n")

In [19]:


def processar_lotes(df: pd.DataFrame, num_lote: int = 5, tamanho_lote: int = 30, modo: str = 'stream', start_offset: int = 0,) -> None:
    total_lotes = num_lote
    arquivo_md = 'resposta_chat_stream.md' if modo == 'stream' else 'resposta_chat.md'
    limite_total = min(len(df), total_lotes * tamanho_lote)
    if limite_total == 0:
        raise ValueError('Nenhum ticket disponível para processamento.')
    if start_offset < 0:
        raise ValueError("start_offset deve ser >= 0")
    
    with open(arquivo_md, 'a', encoding='utf-8') as arquivo_md_saida:
        titulo = 'Respostas ITIL por Lote - Stream' if modo == 'stream' else 'Respostas ITIL por Lote - Normal'
        arquivo_md_saida.write(f'# {titulo}\n\n')
    processed_end = start_offset + limite_total
    for lote_numero in range(total_lotes):
        inicio = start_offset + lote_numero * tamanho_lote
        fim = inicio + tamanho_lote
        if inicio >= processed_end:
            print(f'Lote {lote_numero + 1}: nenhum ticket restante.')
            break

        prompt_lote = gerar_prompt_excluindo_primeiros(
            df,
            excluir=inicio,
            idioma='pt',
            quantidade=tamanho_lote,
        )

        print(f'Processando lote {lote_numero + 1}/{total_lotes} com tickets {inicio + 1} a {min(fim, processed_end)}...')
        if modo == 'stream':
            gerar_resposta_stream(prompt_lote, caminho_md=arquivo_md)
        else:
            gerar_resposta(prompt_lote, caminho_md=arquivo_md)


print('✅ Funções de lote para chat prontas.')      

✅ Funções de lote para chat prontas.


In [20]:
processar_lotes(df_tickets, num_lote=5, tamanho_lote=30, modo='stream',start_offset=150)

Sucesso! Prompt montado para um lote de 30 tickets.
Processando lote 1/5 com tickets 151 a 180...
| Ticket ID | Prática Principal | Prática Auxiliar |
| :--- | :--- | :--- |
| 151 | Gerenciamento de Incidentes | Gerenciamento de Problemas |
| 162 | Gerenciamento de Incidentes | Gerenciamento de Disponibilidade |
| 182 | Gerenciamento de Incidentes | Gerenciamento de Problemas |
| 183 | Gerenciamento Financeiro de Serviços | Gerenciamento de Incidentes |
| 189 | Gerenciamento de Incidentes | Gerenciamento de Problemas |
| 194 | Gerenciamento de Incidentes | Gerenciamento de Problemas |
| 201 | Gerenciamento de Incidentes | Gerenciamento de Problemas |
| 203 | Gerenciamento Financeiro de Serviços | Gerenciamento de Incidentes |
| 205 | Gerenciamento de Requisição de Serviço | Gerenciamento de Catálogo de Serviço |
| 219 | Gerenciamento de Relacionamento | Análise de Negócio |
| 223 | Gerenciamento de Mudanças | Gerenciamento de Configuração de Serviço |
| 225 | Gerenciamento de Requisiçã

In [21]:
# processar_lotes(df_tickets, num_lote=5, tamanho_lote=30, modo='stream')

In [22]:
processar_lotes(df_tickets, num_lote=5, tamanho_lote=30, modo='normal',start_offset=150)

Sucesso! Prompt montado para um lote de 30 tickets.
Processando lote 1/5 com tickets 151 a 180...
| Ticket ID | Prática Principal | Prática Auxiliar |
| :--- | :--- | :--- |
| #151 | Gerenciamento de Incidentes | Gerenciamento de Problemas |
| #162 | Gerenciamento de Incidentes | Gerenciamento de Disponibilidade |
| #182 | Gerenciamento de Incidentes | Gerenciamento de Problemas |
| #183 | Gerenciamento Financeiro de Serviços | Gerenciamento de Incidentes |
| #189 | Gerenciamento de Incidentes | Gerenciamento de Problemas |
| #194 | Gerenciamento de Incidentes | Gerenciamento de Problemas |
| #201 | Gerenciamento de Incidentes | Gerenciamento de Problemas |
| #203 | Gerenciamento Financeiro de Serviços | Gerenciamento de Incidentes |
| #205 | Gerenciamento de Requisição de Serviço | Gerenciamento de Catálogo de Serviço |
| #219 | Gerenciamento de Relacionamento | Gerenciamento de Nível de Serviço |
| #223 | Gerenciamento de Mudanças | Gerenciamento de Configuração de Serviço |
| #225 |

In [23]:
# processar_lotes(df_tickets, num_lote=5, tamanho_lote=30, modo='normal')

In [24]:
# df_tickets.info()

In [25]:
# df_tickets.to_excel("amostra.xlsx", index=False)

In [26]:
# df_tickets.to_csv("amostra.csv", index=False)